In [14]:
import os
import json
from dotenv import load_dotenv
from rich.console import Console

from openai import OpenAI
from agents import Agent, Runner, trace, function_tool

In [ ]:
# List of TODO and COMPLETED #
todos = []
completed = []

load_dotenv(override=True)

def show(text):
    try:
        Console().print(text)
    except Exception:
        print(text)


def get_todo_report() -> str:
    result = ""
    for index, todo in enumerate(todos):
        if completed[index]:
            result += f"Todo #{index + 1}: [green][strike]{todo}[/strike][/green]\n"
        else:
            result += f"Todo #{index + 1}: {todo}\n"
    show(result)
    return result

@function_tool
def create_todos(descriptions: list[str]) -> str:
    """Add new todos from a list of descriptions and return the full list"""
    todos.extend(descriptions)
    completed.extend([False] * len(descriptions))
    return get_todo_report()

@function_tool
def mark_complete(index: int, completion_notes: str) -> str:
    """Mark complete the todo at the given position (starting from 1) and return the full list"""
    if 1 <= index <= len(todos):
        completed[index - 1] = True
    else:
        return "No todo at this index."
    Console().print(completion_notes)
    return get_todo_report()

tools = [create_todos, mark_complete]

openai = OpenAI()

question_generator_prompt = "Please propose a hard, challenging question to assess someone's IQ. Respond only with the question."
messages = [{"role": "user", "content": question_generator_prompt}]
questonGeneratorResponse = openai.chat.completions.create(
    model="gpt-4.1-mini",
    messages=messages
    )

question = question = questonGeneratorResponse.choices[0].message.content

print(f"Generated Question: {question} \n\n")

print(tools)

print("\n\n")

todo_planner_executor_system_prompt = """
You are  a todo list planner and executor. For a given a problem to solve, by using your todo tools to plan a list of steps, then carrying out each step in turn.
Now use the todo list tools, create a plan, carry out the steps, and reply with the solution.
If any quantity isn't provided in the question, then include a step to come up with a reasonable estimate.
Provide your solution in Rich console markup without code blocks.
Do not ask the user questions or clarification; respond only with the answer after using your tools.
"""
todoPlannerAndExecutorAgent = Agent(
    name="Todo List Generator and Executor",
    instructions= todo_planner_executor_system_prompt,
    tools= tools
)

print("Trace Available at: https://platform.openai.com/logs?api=traces")

result = ""

with trace("TO List Planner and Executor Trace"):
    result = await Runner.run(todoPlannerAndExecutorAgent, question)

print(f"\n\n {result}")

Generated Question: A bat and a ball cost $1.10 in total. The bat costs $1.00 more than the ball. How much does the ball cost? 




AttributeError: 'list' object has no attribute 'read'